In [ ]:
# ============================================================
# Konfiguration (zentral & leicht anpassbar)
# Alle Stellschrauben an einem Ort -> kein Suchen mehr quer durch den Code.
# ============================================================

# Reproduzierbarkeit
SEED = 42

# Pfade
# GEAENDERT: NextStep4 -> NextStep5. Mit BatchNorm enthaelt der state_dict
# zusaetzliche Eintraege (running_mean, running_var, num_batches_tracked).
# Wuerden wir die alten Dateien ueberschreiben, waere das NextStep4-Notebook
# nicht mehr ladbar.
DATA_PATH     = '../Datasets/Iris.csv'
MODEL_PATH    = '../Models/iris_net_NextStep5.pth'
ENCODER_PATH  = '../Models/label_encoder_NextStep5.pkl'
SCALER_PATH   = '../Models/scaler_NextStep5.pkl'
METADATA_PATH = '../Models/metadata_NextStep5.json'

# ------------------------------------------------------------
# NEU: Batch Normalization ein-/ausschalten
# ------------------------------------------------------------
# True  -> Linear -> BatchNorm1d -> ReLU  (je versteckter Schicht)
# False -> Linear -> ReLU                 (wie bisher)
# Das ist die einzige Stellschraube fuer den Vergleich: Notebook einmal mit
# True und einmal mit False durchlaufen lassen und die Kurven vergleichen.
USE_BATCHNORM = True

# Architektur
# GEAENDERT: frueher eine versteckte Schicht mit 5 Neuronen.
# Das war fuer Iris voellig ausreichend - aber BatchNorm haette dort nichts
# zu tun gehabt: Die Eingaben sind durch den Scaler bereits standardisiert,
# und bei EINER Schicht verschiebt sich die Verteilung nirgends.
# Das Netz ist jetzt bewusst voellig ueberdimensioniert: 16 Schichten und rund
# 16.000 Parameter fuer 96 Trainings-Samples. Als Modell ist das Unsinn - als
# Demonstration ist es genau richtig, denn erst hier tritt das Problem auf,
# gegen das BN entwickelt wurde: Die Verteilung der Aktivierungen driftet von
# Schicht zu Schicht weiter weg, und ohne Gegenmassnahme kommt unten kaum noch
# ein brauchbares Signal an.
HIDDEN_DIM = 32    # Neuronen je versteckter Schicht
N_HIDDEN   = 16    # Anzahl versteckter Schichten

# Training
BATCH_SIZE    = 12
LEARNING_RATE = 0.01
MAX_EPOCHS    = 100
PATIENCE      = 10     # Epochen ohne Verbesserung bis zum Early-Stopping-Abbruch

# Scaler-Auswahl: 'standard' | 'minmax' | 'robust'
SCALER_TYPE = 'standard'


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
import joblib
import json
from torch.utils.data import TensorDataset, DataLoader
import copy
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# --- Reproduzierbarkeit: alle relevanten Zufallsquellen fixieren ---
# Ohne torch.manual_seed waeren Gewichts-Initialisierung und das Shuffeln im
# DataLoader bei jedem Lauf anders -> nicht reproduzierbare Ergebnisse.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1. Laden des Datensatzes
df = pd.read_csv(DATA_PATH)

# 2. Features (X) und Target (y) trennen
X = df.drop('species', axis=1).values
y = df['species'].values

# Feature-Namen direkt aus den Daten ableiten (verbindliche Reihenfolge fuer die Inferenz!)
feature_names = df.drop('species', axis=1).columns.tolist()

# LabelEncoder sortiert die gefundenen Text-Kategorien standardmaessig alphabetisch
le = LabelEncoder()
y = le.fit_transform(y)

# 3. Aufteilen des Datensatzes (Train / Val / Test) - beide Splits mit demselben SEED
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

# --- Feature Scaling ---
# Fuer neuronale Netze ist Skalierung essenziell fuer die Konvergenz. Gaengige Methoden:
# 1. StandardScaler (Standardisierung): Mittelwert = 0, Standardabweichung = 1 (Standard fuer NNs)
# 2. MinMaxScaler (Normalisierung): Skaliert Werte starr in einen Bereich (meist 0 bis 1)
# 3. RobustScaler: Nutzt Median und Quartile, sehr robust gegenueber Ausreissern (Outliers)
# Die Auswahl erfolgt zentral ueber SCALER_TYPE in der Konfiguration:
#
# WICHTIG zur Abgrenzung: Der Scaler normalisiert die EINGABEN, einmalig, vor
# dem Training. BatchNorm normalisiert die AUSGABEN der versteckten Schichten,
# in jedem einzelnen Trainingsschritt neu. Das eine ersetzt das andere nicht.
if SCALER_TYPE == 'standard':
    scaler = StandardScaler()
elif SCALER_TYPE == 'minmax':
    scaler = MinMaxScaler()
elif SCALER_TYPE == 'robust':
    scaler = RobustScaler()
else:
    raise ValueError(f"Unbekannter SCALER_TYPE: {SCALER_TYPE!r} (erlaubt: 'standard', 'minmax', 'robust')")

# WICHTIG (Vermeidung von Data Leakage): 'fit_transform' NUR auf Trainingsdaten anwenden!
# Der Scaler lernt hier die Parameter (z.B. Mittelwert/Varianz) der Trainingsdaten.
X_train = scaler.fit_transform(X_train)

# Auf Validierungs- und Testdaten NUR 'transform' anwenden!
# Sie werden mit den aus den Trainingsdaten gelernten Parametern skaliert.
X_val  = scaler.transform(X_val)
X_test = scaler.transform(X_test)
# -----------------------------------------

# 4. Umwandeln der skalierten Daten in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

# Dimensionen aus den Daten ableiten (statt hartkodiert -> keine Inkonsistenzen mehr)
input_dim  = X_train.shape[1]    # Anzahl Eingabe-Merkmale
output_dim = len(le.classes_)    # Anzahl Klassen

# TensorDataset und DataLoader erstellen
train_dataset = TensorDataset(X_train, y_train)
# Generator mit Seed -> reproduzierbares Shuffeln in jeder Epoche
g = torch.Generator()
g.manual_seed(SEED)

# GEAENDERT: drop_last
# BatchNorm1d braucht im Trainingsmodus mindestens 2 Samples - aus EINEM Sample
# laesst sich keine Varianz schaetzen, PyTorch wirft dann einen RuntimeError.
# Bleibt am Ende einer Epoche ein Rest-Batch der Groesse 1 uebrig, kracht das
# Training mitten im Lauf. drop_last verwirft diesen Rest.
# Hier konkret: 96 Trainings-Samples / BATCH_SIZE 12 = genau 8 Batches, es geht
# also kein einziges Sample verloren. Bei BATCH_SIZE = 19 saehe das anders aus:
# 5 * 19 = 95, ein Sample bliebe uebrig -> genau der Absturz.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=g, drop_last=USE_BATCHNORM)


# ============================================================
# Definition des Netzes
# ============================================================
def baue_netz(input_dim, hidden_dim, n_hidden, output_dim, use_batchnorm):
    """Baut das Netz an EINER zentralen Stelle.

    Die Wiederlade-Zelle weiter unten benutzt dieselbe Funktion. Im
    urspruenglichen Notebook stand die Architektur zweimal im Code - mit
    BatchNorm waere das eine sichere Fehlerquelle: Der state_dict enthaelt
    dann zusaetzlich running_mean, running_var und num_batches_tracked, und
    load_state_dict() bricht ab, sobald die beiden Definitionen auseinanderlaufen.
    """
    schichten = []
    prev_dim = input_dim

    for _ in range(n_hidden):
        # bias=False bei aktivem BN: BatchNorm zieht den Mittelwert unmittelbar
        # danach wieder ab, der Bias waere also wirkungslos. Seine Rolle
        # uebernimmt der lernbare Parameter beta innerhalb der BN-Schicht.
        schichten.append(nn.Linear(prev_dim, hidden_dim, bias=not use_batchnorm))

        if use_batchnorm:
            # Der eigentliche Einbau: VOR der Aktivierungsfunktion.
            # Normalisiert spaltenweise, also je Neuron ueber alle Samples des Batches.
            schichten.append(nn.BatchNorm1d(hidden_dim))

        schichten.append(nn.ReLU())
        prev_dim = hidden_dim

    # Ausgabeschicht: hier KEIN BN. Die Logits gehen direkt in die
    # CrossEntropyLoss, ihre Skala soll das Netz frei bestimmen koennen.
    schichten.append(nn.Linear(prev_dim, output_dim))

    return nn.Sequential(*schichten)


net = baue_netz(input_dim, HIDDEN_DIM, N_HIDDEN, output_dim, USE_BATCHNORM)

anzahl_parameter = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Variante: {'MIT' if USE_BATCHNORM else 'OHNE'} BatchNorm | "
      f"{N_HIDDEN} versteckte Schichten a {HIDDEN_DIM} Neuronen | "
      f"{anzahl_parameter} trainierbare Parameter")

# Definieren des Verlustkriteriums und des Optimierers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=LEARNING_RATE)

# --- Setup fuer Early Stopping ---
patience_counter = 0          # Zaehler fuer die Epochen ohne Verbesserung
best_val_loss = float('inf')  # bester bisher gesehener Validation Loss (als float)
best_model_weights = None     # hier speichern wir die besten Gewichte

# Listen zum Speichern der Historie (ideal fuer spaetere Plots, z.B. mit matplotlib)
history = {'train_loss': [], 'val_loss': []}

# Setup der Figur VOR der Schleife
fig, ax = plt.subplots(figsize=(8, 6))

# Trainieren des neuronalen Netzes
for epoch in range(MAX_EPOCHS):
    net.train()  # Trainingsmodus aktivieren (relevant fuer Dropout/BatchNorm)
    kumulierter_train_loss = 0.0  # Variable zum Aufsummieren des Batch-Losses

    for batch_data in train_loader:  # Schleife ueber die Batches aus dem DataLoader
        batch_X = batch_data[0]
        batch_y = batch_data[1]
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        # Loss aufsummieren
        kumulierter_train_loss += loss.item()

    durchschnittlicher_train_loss = kumulierter_train_loss / len(train_loader)
    history['train_loss'].append(durchschnittlicher_train_loss)

    # Validierung nach jeder Epoche
    # net.eval() ist mit BatchNorm nicht mehr optional, sondern zwingend:
    # Im Eval-Modus benutzt BN die mitgefuehrten running-Statistiken statt der
    # Statistik des aktuellen Batches. Sonst haenge die Vorhersage fuer ein
    # Sample davon ab, welche anderen Samples zufaellig daneben liegen.
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val).item()  # direkt als float speichern

    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch:3d} | Loss: {durchschnittlicher_train_loss:.4f} | Val Loss: {val_loss:.4f}')

        # --- Live Plotting in Jupyter ---
    ax.clear() # Löscht die alten Linien im Plot-Objekt

    ax.plot(history['train_loss'], label='Train Loss', color='blue', linewidth=2)
    ax.plot(history['val_loss'],   label='Val Loss', color='orange', linewidth=2)

    ax.set_xlabel('Epoche')
    ax.set_ylabel('Loss')
    ax.set_title(f"Training & Validation Loss (Epoche {epoch+1}) - "
                 f"{'mit' if USE_BATCHNORM else 'ohne'} BatchNorm")
    ax.set_xlim(0, 100) 
    ax.set_ylim(bottom=0) # Lässt Matplotlib das obere Limit automatisch anpassen!
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

    # Der Jupyter-Magic-Trick:
    clear_output(wait=True) # Löscht den alten Output der Zelle (wait=True verhindert Flackern)
    display(fig)            # Zeichnet die aktualisierte Figur
    # --------------------------------

    # --- Early Stopping Logik ---
    # Ist der aktuelle Validation Loss der beste, den wir je gesehen haben?
    # Hinweis: state_dict() enthaelt bei BatchNorm auch die Buffer running_mean
    # und running_var. deepcopy sichert sie korrekt mit - Early Stopping stellt
    # also nicht nur die Gewichte, sondern auch die Inferenz-Statistik wieder her.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0  # Zaehler zuruecksetzen
        best_model_weights = copy.deepcopy(net.state_dict())
    else:
        patience_counter += 1  # Keine Verbesserung -> Zaehler erhoehen

    # Zu lange keine Verbesserung -> Abbruch!
    if patience_counter >= PATIENCE:
        print(f"\n--- Early Stopping ausgeloest in Epoche {epoch+1} ---")
        print(f"Bester Validation Loss war: {best_val_loss:.4f}")
        break
    # ----------------------------------------------

# Wir laden die besten Gewichte zurueck in das Modell.
# Ohne diesen Schritt haetten wir das Modell im Zustand des Overfittings!
net.load_state_dict(best_model_weights)
print("\nBeste Modellgewichte wurden wiederhergestellt.")


# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)

# --- Abspeichern: Modell, Encoder, Scaler + Metadaten ---
torch.save(net.state_dict(), MODEL_PATH)
joblib.dump(le, ENCODER_PATH)
joblib.dump(scaler, SCALER_PATH)

# Metadaten beschreiben, WIE das Modell zu benutzen ist: Feature-Reihenfolge,
# Klassen-Mapping, Skalierungstyp, Architektur und Version. Ohne sie muss
# beim spaeteren Laden alles erraten werden.
# GEAENDERT: n_hidden und use_batchnorm gehoeren jetzt zwingend dazu - ohne sie
# laesst sich das Netz nicht mehr rekonstruieren und load_state_dict() scheitert.
metadata = {
    'feature_names': feature_names,
    'class_names': le.classes_.tolist(),
    'class_mapping': {int(i): name for i, name in enumerate(le.classes_)},
    'scaler_type': SCALER_TYPE,
    'architecture': {
        'input_dim': int(input_dim),
        'hidden_dim': int(HIDDEN_DIM),
        'n_hidden': int(N_HIDDEN),
        'output_dim': int(output_dim),
        'use_batchnorm': bool(USE_BATCHNORM),
    },
    'best_val_loss': float(best_val_loss),
    'test_accuracy': float(accuracy),
    'epochs_trained': int(epoch + 1),
    'seed': SEED,
    'torch_version': torch.__version__,
}
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("Metadaten gespeichert:", METADATA_PATH)


In [ ]:
# Wiederladen von Modell + Artefakten (aus den Metadaten rekonstruiert)
import torch
import torch.nn as nn
import joblib
import json

# Metadaten ZUERST laden -> daraus die Architektur rekonstruieren
with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
arch = metadata['architecture']

# WICHTIG: Erst die Architektur definieren, DANN die Gewichte laden.
# (Im urspruenglichen Skript war die Reihenfolge vertauscht: die geladenen
#  Gewichte wurden sofort durch ein neu initialisiertes Netz ueberschrieben.)
#
# GEAENDERT: Hier stand die Architektur frueher ein zweites Mal ausgeschrieben.
# Mit BatchNorm geht das nicht mehr gut: der state_dict enthaelt jetzt auch
# running_mean/running_var, und schon eine kleine Abweichung zwischen den
# beiden Definitionen laesst load_state_dict() abbrechen. Deshalb benutzen wir
# dieselbe Funktion wie im Training und lesen alle Parameter aus den Metadaten.
net = baue_netz(
    input_dim     = arch['input_dim'],
    hidden_dim    = arch['hidden_dim'],
    n_hidden      = arch['n_hidden'],
    output_dim    = arch['output_dim'],
    use_batchnorm = arch['use_batchnorm'],
)
net.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))

# net.eval() schaltet BatchNorm auf die gespeicherten running-Statistiken um.
# Ohne diesen Aufruf wuerde jede Vorhersage mit der Statistik der zufaellig
# mitgeschickten Samples rechnen.
net.eval()

le = joblib.load(ENCODER_PATH)
scaler = joblib.load(SCALER_PATH)

# named_parameters() zeigt nur die LERNBAREN Parameter - bei BatchNorm also
# weight (= gamma) und bias (= beta), je einer pro Neuron.
for k, v in net.named_parameters():
    print(k, tuple(v.shape))

# running_mean und running_var sind KEINE Parameter, sondern Buffer. Sie werden
# nicht per Gradientenabstieg gelernt, sondern waehrend des Trainings
# mitgeschrieben. Trotzdem stecken sie im state_dict und im gespeicherten Modell.
print("\nBuffer (nicht lernbar, aber gespeichert):")
for k, v in net.named_buffers():
    print(k, tuple(v.shape))


In [ ]:
# Vorhersage mit dem trainierten Netz
# Statt interaktivem input() definieren wir die Eingaben als Array. Das ist
# reproduzierbar, skript-/batch-tauglich und leicht erweiterbar: einfach
# weitere Zeilen ergaenzen. Die Spalten-Reihenfolge MUSS metadata['feature_names'] entsprechen.
import numpy as np

print("Erwartete Merkmal-Reihenfolge:", metadata['feature_names'])

# Beispiel-Eingaben (eine Zeile = ein Datenpunkt)
beispiel_eingaben = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.7, 3.0, 5.2, 2.3],
])

# Skalierung mit demselben (geladenen) Scaler wie im Training
eingabe_skaliert = scaler.transform(beispiel_eingaben)

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor(eingabe_skaliert, dtype=torch.float32)

# Vorhersage treffen
# Das Netz steht dank net.eval() im Inferenzmodus. Nur deshalb funktioniert
# dieser Aufruf mit lediglich 2 Datenpunkten - im Trainingsmodus wuerde
# BatchNorm die Statistik aus genau diesen 2 Zeilen berechnen.
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

# Ausgabe der Klassifizierungsaussage je Datenpunkt
for i, idx in enumerate(predicted):
    klasse = le.inverse_transform([idx.item()])[0]
    print(f"Eingabe {beispiel_eingaben[i].tolist()} -> Klasse: {klasse}")


In [ ]:
# Vorhersagen als Wahrscheinlichkeiten (nutzt 'inputs' aus der vorherigen Zelle)
import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)

for i in range(len(inputs)):
    klasse = le.inverse_transform([max_index[i].item()])[0]
    wahrscheinlichkeiten = {
        le.classes_[c]: round(float(probabilities[i, c]), 4)
        for c in range(len(le.classes_))
    }
    print(f"Datenpunkt {i}: Klasse = {klasse} | Wahrscheinlichkeiten = {wahrscheinlichkeiten}")
